# 05 · Previsões com o modelo vigente

**Feature F8** (critério 8) · dono: ver `config/equipe.yaml` · funções em `src/promocao.py`

| | |
|---|---|
| **Entradas** | `models/modelo_vigente.json` e o arquivo do vigente em `models/registro/`, o próximo lote de `data/incoming/` |
| **Saídas** | `evidencias/previsoes/previsoes_<cycle_id>.csv` · conferência das previsões de ciclos anteriores · `evidencias/previsoes/demonstracao_<cycle_id>.csv` (demonstração com reservas novas) |

**Regras**
- O vigente é o `Pipeline` completo da F5: pré-processamento, seleção das features eleitas e algoritmo.
- As previsões são feitas no **próximo lote de `data/incoming/`**: reservas que ainda não entraram no pipeline e não participaram da escolha do modelo (o teste do ciclo já foi usado na promoção, no notebook 04).
- O lote passa pelo mesmo ETL do notebook 01. O lote simulado traz o rótulo, porque o dataset é histórico, mas ele não é usado na previsão nem gravado com ela.
- Quando o lote chega no ciclo seguinte, as previsões são conferidas com o rótulo real.
- Sem lote restante em `data/incoming/` (os 8 lotes de 2017 acabam depois de 8 ciclos), o notebook avisa e mostra as **últimas previsões feitas**, identificadas pelo ciclo que as gerou. Nada novo é gravado nesse caso.
- Sem vigente, o notebook só avisa e termina sem erro.
- A demonstração no fim prevê reservas novas, sem rótulo, com `src/prever.py` (o mesmo de `python -m src.prever`). Ela roda em todo ciclo, mesmo sem lote novo.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, carregar_contexto, resolver
from src.preprocessamento import carregar_particao
from src import prever, promocao

cfg = carregar_config()
ctx = carregar_contexto(cfg, RAIZ)  # escrito pelo notebook 01
print(f"cycle_id={ctx['cycle_id']} | data_version={ctx['data_version']}")

## 1. Modelo vigente

In [ ]:
manifesto = promocao.carregar_manifesto(cfg, RAIZ)
modelo, vigente = promocao.carregar_vigente(manifesto, RAIZ)
if vigente is None:
    print("AVISO: não há modelo vigente (nenhum ciclo promoveu ainda). Nada a prever.")
else:
    print(f"vigente: modelo {vigente['modelo']} ({vigente['algoritmo']}) do ciclo {vigente['cycle_id']}")
    print(f"data_version do treino: {vigente['data_version']} | {len(vigente['features'])} features | arquivo: {vigente['arquivo']}")

## 2. Previsões no próximo lote

In [ ]:
arquivo_lote = promocao.proximo_lote(cfg, RAIZ)
previsoes = None
if vigente is None:
    pass
elif arquivo_lote is None:
    print("AVISO: não há mais lotes em data/incoming. Sem lote futuro para prever neste ciclo; nada novo foi gravado.")
    caminho_ultimas, ultimas = promocao.ultimas_previsoes(RAIZ)
    if ultimas is None:
        print("Nenhuma previsão anterior gravada.")
    else:
        tempo = cfg["avaliacao"]["coluna_tempo"]
        print(f"Últimas previsões feitas: {caminho_ultimas.name} (ciclo {ultimas['cycle_id'].iloc[0]}, "
              f"vigente {ultimas['vigente_modelo'].iloc[0]} do ciclo {ultimas['vigente_cycle_id'].iloc[0]})")
        print(f"lote {ultimas['lote_id'].iloc[0]} | {len(ultimas):,} reservas | chegadas de {ultimas[tempo].min()} a {ultimas[tempo].max()}")
        print(f"cancelamento previsto: {ultimas['previsao'].mean():.1%} | probabilidade média: {ultimas['prob_cancelamento'].mean():.3f}")
else:
    lote, rel = promocao.preparar_lote(arquivo_lote, cfg)
    previsoes = promocao.prever(modelo, lote, cfg)
    caminho = promocao.salvar_previsoes(previsoes, ctx, vigente, RAIZ)
    tempo = cfg["avaliacao"]["coluna_tempo"]
    print(f"lote: {arquivo_lote.name} | {rel['n_entrada']:,} linhas brutas, {rel['removidas_total']:,} removidas pelo ETL, {len(previsoes):,} previstas")
    print(f"chegadas de {previsoes[tempo].min():%Y-%m-%d} a {previsoes[tempo].max():%Y-%m-%d}")
    print(f"cancelamento previsto: {previsoes['previsao'].mean():.1%} | probabilidade média: {previsoes['prob_cancelamento'].mean():.3f}")
    print("salvo em", caminho.relative_to(RAIZ))
    display(previsoes.head())

## 3. Conferência das previsões anteriores

Previsões de ciclos anteriores cujo lote já entrou no pipeline, comparadas com o rótulo real.

In [ ]:
conferencia = promocao.conferir_previsoes_anteriores(cfg, RAIZ)
if conferencia.empty:
    print("Nenhuma previsão anterior com lote já incorporado para conferir.")
else:
    display(conferencia.round(4))

## 4. O que mudou entre os ciclos

In [ ]:
historico = promocao.tabela_historico(manifesto)
if historico.empty:
    print("Nenhuma decisão registrada ainda.")
else:
    display(historico.round(4))

## 5. Demonstração de uso: reservas novas

O uso final do modelo: a gestão de reservas informa uma reserva que acabou de entrar e recebe a probabilidade de cancelamento. A entrada traz só o que se sabe no ato da reserva, sem o desfecho (`is_canceled`) e sem as colunas posteriores (`assigned_room_type`, `booking_changes`, `reservation_status`). As reservas de exemplo estão em `exemplos/reservas_exemplo.json`.

O mesmo resultado sai na linha de comando: `python -m src.prever --json exemplos/reserva_exemplo.json`.

In [ ]:
if vigente is None:
    print("Sem vigente: nada a demonstrar.")
else:
    reservas = prever.ler_reservas(RAIZ / prever.EXEMPLO)
    demonstracao = prever.prever_reserva(reservas, modelo, vigente, cfg, RAIZ)
    print(prever.descrever(demonstracao))
    print("salvo em", prever.salvar_demonstracao(demonstracao, ctx, RAIZ).relative_to(RAIZ))
    display(demonstracao)